# 02 Collect Sources

Collect and audit the dynamic source artifacts used by the Marshfield Event Catalog: CORA coastal water level, ERA5 ocean waves, direct AORC SST rainfall members, and NWM soil-moisture context.


> Requires: region setup outputs, data_sources.yaml, Earthdata credentials where required
>
> Produces: CORA, ERA5 waves, AORC SST rainfall, NWM soil-moisture source artifacts

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

location_root = Path("..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

# plan reruns, reuse reviewed data, and audit readiness.
import design_events.collect_sources.workflow as collect
from design_events.utils import load_runtime as load_design_event_runtime

config, paths = load_design_event_runtime(location_root / "config.yaml")
display(collect.summary(config, paths))

## Rerun Control


In [ ]:
rerun = True

## Source Collection Plan

In [ ]:
plan = collect.plan(config, paths)
plan_table = collect.plan_table(
    plan,
    paths,
    rerun=rerun,
)
display(plan_table)


## CORA Coastal Water Level

CORA provides the historical boundary water-level series for Marshfield surge-event fitting. The downstream Event Catalog expects the hourly MSL CSV at `paths["waterlevel_csv"]`.


In [ ]:
cora_path = paths["waterlevel_csv"]
if cora_path.exists():
    waterlevel = pd.read_csv(cora_path, parse_dates=["time"])
    display(pd.Series({"rows": len(waterlevel), "start": waterlevel["time"].min(), "end": waterlevel["time"].max(), "path": str(cora_path)}))
    display(waterlevel.head())
else:
    display(pd.Series({"status": "missing", "expected_path": str(cora_path)}))


## ERA5 Ocean Waves

Marshfield is wave-enabled. The configured Earth Data Hub ERA5 ocean-wave Zarr variables are staged into a local NetCDF for SnapWave/SFINCS coupling.

In [ ]:
waves = config["collection"].get("era5_waves", {})
display(pd.Series({
    "provider": waves.get("provider"),
    "bbox_wgs84": waves.get("bbox_wgs84"),
    "output": str(paths["era5_waves_nc"]),
    "exists": paths["era5_waves_nc"].exists(),
}))


## Stochastic Storm Transposition Region

The SST region is the configured AORC SST transposition polygon, not a notebook-specific hand draw. For Marshfield, `config.yaml` points to the 100 km candidate region around the study grid footprint, chosen during the AORC homogeneity review so the rainfall catalogue samples plausible coastal New England storms without drifting into a much broader regional climate.


In [ ]:
# Plot the configured AORC SST transposition region before pulling rainfall members.
fig, ax = collect.plot_sst_region(config, paths)

## Direct AORC SST Rainfall Members

The direct AORC SST collector scans the transposition region, ranks rolling storm windows, and writes the rainfall-member table for precipitation pairing with synthetic coastal water-level events.


In [ ]:
# --- AORC SST collection parameters (edit to retune, then run Run Collection below) ---
# Threshold-driven POT: keep every INDEPENDENT storm whose 72h footprint-mean depth
# exceeds the threshold, so the rainfall-member count is data-driven
# Set the threshold from the rainfall POT diagnostics; raise it for fewer, heavier members.
min_precip_threshold = 60.0   # mm over the 72h storm window (footprint mean)
decluster_hours = 72          # minimum spacing between independent storms
storm_duration_hours = 72     # SST rainfall accumulation window
check_every_n_hours = 6       # transposition scan stride

display(collect.aorc_sst_params(
    config,
    paths,
    min_precip_threshold=min_precip_threshold,
    decluster_hours=decluster_hours,
    storm_duration_hours=storm_duration_hours,
    check_every_n_hours=check_every_n_hours,
))

## NWM Soil-Moisture Context

Marshfield has no meaningful streamflow boundary in the baseline, but selected NWM LDAS soil-moisture cells are retained for antecedent-condition pairing.


In [ ]:
display(collect.soil_sources(config, paths))


## Run Collection

Run each configured source inside this notebook and show progress as each provider finishes. This cell is configured for the full Marshfield production collection window, so it overwrites partial smoke artifacts instead of reusing them.


In [ ]:
# Collect any missing configured sources and summarize the artifacts.
collection_result_table = collect.run_collect(
    config,
    paths,
    plan,
    run_collection=True,
    skip_existing=not rerun,
)
display(collection_result_table)


## Collected Data Overview

Plot the collected source data in the form that best matches each source: spatial footprints and sample points for geography, source-window bars for coverage, and time-series or distribution plots for the dynamic forcing records.


### SST

Storm transposition targets are plotted against the configured SST region and study area.


In [ ]:
# Plot the SST region, study area, and rainfall transposition targets.
fig, ax = collect.plot_collected_sst_geography(config, paths)

### CORA Boundary Water Level

Boundary water levels are plotted independently from the spatial source map.


In [ ]:
# Plot the collected CORA boundary water level.
fig, ax = collect.plot_cora_boundary_water_level(paths)

### NWM Soil Moisture

Soil moisture is summarized across configured NWM points and layers.


In [ ]:
# Plot monthly mean NWM soil moisture variables when available.
fig, status = collect.plot_nwm_soil_moisture(config, paths)
display(status)

### AORC SST Rainfall

Rainfall members are shown as selected-event magnitude and distribution summaries.


In [ ]:
# Plot compact AORC SST rainfall member summaries.
fig = collect.plot_aorc_sst_rainfall(paths)

### ERA5 Wave Forcing

Wave variables are reduced to monthly spatial means for a compact quality check.


In [ ]:
# Plot compact ERA5 wave time series when available.
fig = collect.plot_era5_waves(paths)